# TP4 — Transformers & NLP
## Introduction aux Transformers avec Hugging Face — Pipelines, BERT, GPT, NER, QA
**Étudiant :** BARHOINE AYOUB | **Filière :** FIGI 2ème Année | **Module :** Deep Learning  
**Framework :** Hugging Face Transformers v5.0.0 + Datasets v4.0.0 | **Référence :** NLP with Transformers — O'Reilly (Tunstall et al.)

## Table des Matières
1. Introduction aux Transformers
2. Environnement et Configuration
3. Architecture des Transformers
4. Tokenisation
5. Pipelines Hugging Face — Vue d'ensemble
6. Pipeline 1 — Classification de Texte (Sentiment Analysis)
7. Pipeline 2 — Named Entity Recognition (NER)
8. Pipeline 3 — Question Answering
9. Autres Pipelines Hugging Face
10. Comparaison des Modèles Transformers
11. Conclusion

## 1. Introduction aux Transformers

Les Transformers représentent une révolution dans le NLP. Introduits en 2017 par Vaswani et al. dans *"Attention Is All You Need"*, ils ont remplacé les RNN/LSTM grâce à leur mécanisme d'**attention multi-têtes** qui traite les séquences en parallèle et capture les dépendances à longue portée.

| Année | Modèle | Innovation |
|-------|--------|------------|
| 2017 | Transformer (Vaswani) | Attention multi-têtes, architecture encodeur-décodeur |
| 2018 | BERT (Google) | Pré-entraînement bidirectionnel (Masked LM + NSP) |
| 2019 | GPT-2 (OpenAI) | Génération de texte autoregressive, decoder-only |
| 2019 | DistilBERT (HuggingFace) | BERT distillé : 40% plus léger, 60% plus rapide, 97% des perfs |
| 2019 | XLM-RoBERTa | BERT multilingue sur 100 langues |
| 2019 | T5 (Google) | Text-to-Text : toutes tâches NLP en un seul modèle |
| 2020 | GPT-3 (OpenAI) | 175B paramètres, few-shot learning remarquable |

## 2. Environnement et Configuration

In [ ]:
# Installation des bibliothèques Hugging Face
!pip install transformers datasets accelerate sentencepiece -q

In [ ]:
import transformers
import datasets as hf_datasets
import torch
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print(f'Using transformers {transformers.__version__}')
print(f'Using datasets {hf_datasets.__version__}')
print(f'PyTorch : {torch.__version__}')
print(f'GPU disponible : {torch.cuda.is_available()}')

## 3. Architecture des Transformers

Un Transformer est composé d'un **encodeur** et d'un **décodeur**, chacun constitué de couches identiques empilées. Le mécanisme clé est l'**attention multi-têtes** (Multi-Head Attention).

### 3.1 Les trois familles de modèles
| Famille | Architecture | Modèles | Usage |
|---------|-------------|---------|-------|
| Encoder-only | Encodeur seul | BERT, DistilBERT, RoBERTa, XLM-R | Classification, NER, QA extractif |
| Decoder-only | Décodeur seul | GPT-2, GPT-3, GPT-4 | Génération de texte |
| Encoder-Decoder | Encodeur + Décodeur | T5, BART, mT5 | Traduction, résumé |

### 3.2 Mécanisme d'Attention
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

L'attention multi-têtes applique ce mécanisme en parallèle avec h=8 ou h=16 têtes différentes.

## 4. Tokenisation

In [ ]:
from transformers import AutoTokenizer

# Chargement du tokenizer DistilBERT
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

text = "I love deep learning!"
tokens = tokenizer.tokenize(text)
input_ids = tokenizer.encode(text)

print(f'Texte original : "{text}"')
print(f'Tokens         : {tokens}')
print(f'Token IDs      : {input_ids}')
print(f'Nb de tokens   : {len(tokens)}')
print('\nTokens spéciaux :')
print(f'  [CLS] = {tokenizer.cls_token_id}  → début de séquence, utilisé pour classification')
print(f'  [SEP] = {tokenizer.sep_token_id}  → séparateur / fin de séquence')
print('\nNote : "learn" + "##ing" = tokenisation sous-mot (WordPiece) de DistilBERT')

# Tableau des algorithmes de tokenisation
tokenizers_info = {
    'Algorithme': ['WordPiece', 'BPE', 'SentencePiece', 'Unigram'],
    'Modèles': ['BERT, DistilBERT', 'GPT-2, RoBERTa', 'T5, XLM-RoBERTa', 'ALBERT'],
    'Vocabulaire': ['30 000', '50 000', '32 000–250 000', '30 000'],
    'Particularité': ['Tokens ## pour suffixes', 'Basé sur fréquences de paires',
                      'Multilingue, pas d\'espace fixe', 'Probabiliste']
}
print('\nAlgorithmes de tokenisation :')
print(pd.DataFrame(tokenizers_info).to_string(index=False))

## 5. Pipelines Hugging Face — Vue d'ensemble

In [ ]:
from transformers import pipeline

print('L\'API pipeline() fournit une interface unifiée pour utiliser des modèles pré-entraînés.')
print('En une seule ligne de code : tokenisation + inférence + post-traitement.\n')

pipelines_list = {
    'Pipeline': [
        'text-classification', 'ner', 'question-answering',
        'text-generation', 'summarization', 'translation',
        'fill-mask', 'zero-shot-classification'
    ],
    'Tâche': [
        'Analyse de sentiment', 'NER', 'Question-Réponse',
        'Génération de texte', 'Résumé automatique', 'Traduction',
        'Complétion de masque', 'Classification zero-shot'
    ],
    'Modèle par défaut': [
        'distilbert-base-uncased-finetuned-sst-2-english',
        'dbmdz/bert-large-cased-finetuned-conll03-english',
        'distilbert-base-cased-distilled-squad',
        'gpt2', 'facebook/bart-large-cnn',
        'Helsinki-NLP/opus-mt-en-fr',
        'bert-base-uncased', 'facebook/bart-large-mnli'
    ]
}
print(pd.DataFrame(pipelines_list).to_string(index=False))

## 6. Pipeline 1 — Classification de Texte (Sentiment Analysis)

In [ ]:
# Pipeline text-classification — DistilBERT fine-tuné sur SST-2
print('Chargement du pipeline text-classification (DistilBERT SST-2)...')
classifier = pipeline('text-classification')
print('✅ Modèle chargé')

In [ ]:
# Texte de test — critique négative du film Transformers
text = (
    "This is the worst movie I have ever seen. "
    "The plot is terrible, the acting is bad, and the special effects are laughable. "
    "I cannot recommend this film to anyone."
)

outputs = classifier(text)
df_out  = pd.DataFrame(outputs)
print(df_out)

print(f"\n➡  Label prédit : {outputs[0]['label']}")
print(f"   Score de confiance : {outputs[0]['score']:.4f} ({outputs[0]['score']*100:.2f}%)")
print("\nInterprétation : score >0.5 indique une confiance élevée dans la prédiction")

# Comparaison DistilBERT vs BERT
comparison = {
    'Aspect': ['Paramètres', 'Couches Transformer', 'Hidden size', 'Vitesse', 'Performance SST-2'],
    'DistilBERT': ['66M', '6', '768', '2× plus rapide', '91.3%'],
    'BERT-base':  ['110M', '12', '768', 'Référence', '93.5%']
}
print('\nComparaison DistilBERT vs BERT-base :')
print(pd.DataFrame(comparison).to_string(index=False))

In [ ]:
# Test sur plusieurs phrases
texts = [
    "I absolutely loved this movie, it was fantastic!",
    "The service was terrible and I wasted my money.",
    "The product is okay, nothing special.",
    "Best purchase I have ever made, highly recommend!"
]

results = classifier(texts)
df_multi = pd.DataFrame(results)
df_multi['text'] = [t[:50] + '...' for t in texts]
df_multi = df_multi[['text', 'label', 'score']]
df_multi['score'] = df_multi['score'].map('{:.4f}'.format)
print('Résultats multi-phrases :')
print(df_multi.to_string(index=False))

## 7. Pipeline 2 — Named Entity Recognition (NER)

In [ ]:
# Pipeline NER — BERT-large fine-tuné sur CoNLL-2003
print('Chargement du pipeline NER (BERT-large-cased-CoNLL03)...')
ner = pipeline('ner', aggregation_strategy='simple')
print('✅ Modèle NER chargé')

In [ ]:
# Texte d'analyse — sur le film Transformers
text_ner = (
    "Amazon started selling the movie Transformers in Germany. "
    "Optimus Prime is a great character but Megatron is the villain. "
    "Decept##icons are terrible but Bumblebee is everyone's favorite robot."
)

entities = ner(text_ner)
df_ner = pd.DataFrame(entities)
print(f'Nb entités détectées : {len(entities)}\n')
print(df_ner[['entity_group', 'score', 'word', 'start', 'end']].to_string(index=True))

print('\nAnalyse des entités détectées :')
for e in entities:
    print(f"  [{e['entity_group']:4s}] {e['word']:<20s} score={e['score']:.3f}")

print('\nObservation : Germany (LOC) → confiance ~1.0 (LOC très distinctif)')
print('Optimus Prime, Megatron → MISC (personnages fictifs, absents de CoNLL-2003)')
print('Bumblebee → PER (erreur attendue : c\'est un robot, pas une personne)')

In [ ]:
# Test NER sur un texte personnalisé
text_custom = (
    "Elon Musk founded SpaceX in California and Tesla in Texas. "
    "The company OpenAI is based in San Francisco."
)

entities_custom = ner(text_custom)
df_custom = pd.DataFrame(entities_custom)
print('NER sur texte personnalisé :')
print(df_custom[['entity_group', 'score', 'word']].to_string(index=False))

## 8. Pipeline 3 — Question Answering

In [ ]:
# Pipeline QA — DistilBERT fine-tuné sur SQuAD
print('Chargement du pipeline question-answering (DistilBERT-SQuAD)...')
reader = pipeline('question-answering')
print('✅ Modèle QA chargé')

In [ ]:
# Contexte et question
context = """
The Transformers movie, directed by Michael Bay, was released in 2007. 
It was based on the Hasbro toy franchise. Amazon distributed the movie in Germany 
and in several other countries. The sequel involved an exchange of Megatron with 
the AllSpark, which caused major destruction. Bumblebee, Optimus Prime and the 
other Autobots had to fight against the Decepticons led by Megatron.
"""

question = "What was involved in the sequel of Transformers?"

output_qa = reader(question=question, context=context)
df_qa = pd.DataFrame([output_qa])
print(f'Question : {question}')
print(df_qa.to_string(index=False))

print(f"\nRéponse extraite : '{output_qa['answer']}'")
print(f"Score de confiance : {output_qa['score']:.6f}")
print(f"Position dans le texte : [{output_qa['start']} : {output_qa['end']}]")
print('\nFonctionnement : le modèle produit 2 distributions de probabilité (start, end)')
print('La réponse est le span [start:end] avec la probabilité cumulée maximale.')

In [ ]:
# Test QA sur plusieurs questions
questions = [
    "Who directed the Transformers movie?",
    "When was Transformers released?",
    "Where did Amazon distribute the movie?",
    "Who led the Decepticons?"
]

print('Questions-Réponses multiples :\n')
for q in questions:
    ans = reader(question=q, context=context)
    print(f'Q : {q}')
    print(f"A : '{ans['answer']}' (score={ans['score']:.3f})\n")

## 9. Autres Pipelines Hugging Face

In [ ]:
# Pipeline 4 — Génération de texte (GPT-2)
print('=== Pipeline : text-generation (GPT-2) ===')
generator = pipeline('text-generation', model='gpt2')
outputs_gen = generator('Transformers are', max_new_tokens=50, num_return_sequences=1,
                        pad_token_id=50256, do_sample=True, temperature=0.7)
print(f"Input : 'Transformers are'")
print(f"Output : {outputs_gen[0]['generated_text']}")

In [ ]:
# Pipeline 5 — Fill-Mask (BERT)
print('=== Pipeline : fill-mask (BERT) ===')
unmasker = pipeline('fill-mask', model='bert-base-uncased')
results_mask = unmasker('Deep learning is a [MASK] of machine learning.')
print('Texte : "Deep learning is a [MASK] of machine learning."')
for r in results_mask[:3]:
    print(f"  Token : '{r['token_str']:<15s}' → score={r['score']:.4f}")

In [ ]:
# Pipeline 6 — Résumé automatique (BART)
print('=== Pipeline : summarization (BART-large-CNN) ===')
summarizer = pipeline('summarization', model='facebook/bart-large-cnn')
long_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the 
natural intelligence displayed by animals including humans. AI research has been defined as 
the field of study of intelligent agents, which refers to any system that perceives its 
environment and takes actions that maximize its chance of achieving its goals.
The term AI was coined by John McCarthy in 1956. The field was founded on the assumption 
that human intelligence can be so precisely described that a machine can be made to simulate it.
Modern AI techniques include machine learning, deep learning, and neural networks.
"""
summary = summarizer(long_text, max_length=80, min_length=30, do_sample=False)
print(f'Résumé : {summary[0]["summary_text"]}')

In [ ]:
# Pipeline 7 — Zero-Shot Classification
print('=== Pipeline : zero-shot-classification ===')
zs_classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
text_zs = 'The new iPhone 16 has an amazing camera and battery life.'
candidate_labels = ['technology', 'sports', 'politics', 'entertainment', 'science']
result_zs = zs_classifier(text_zs, candidate_labels=candidate_labels)
print(f'Texte : "{text_zs}"')
print('Labels et scores :')
for label, score in zip(result_zs['labels'], result_zs['scores']):
    print(f'  {label:<15s} : {score:.4f}')

## 10. Comparaison des Modèles Transformers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Tableau de comparaison
models_comparison = {
    'Modèle': ['BERT', 'DistilBERT', 'GPT-2', 'T5', 'RoBERTa', 'XLM-RoBERTa'],
    'Type': ['Encoder-only', 'Encoder-only', 'Decoder-only', 'Encoder-Decoder', 'Encoder-only', 'Encoder-only'],
    'Paramètres': ['110M', '66M', '117M-1.5B', '60M-11B', '125M', '125M-560M'],
    'Tâche principale': ['Classification, NER, QA', 'Classification (rapide)', 'Génération de texte',
                         'Résumé, Traduction', 'Classification améliorée', 'NER Multilingue'],
    'Année': [2018, 2019, 2019, 2019, 2019, 2019]
}
df_comp = pd.DataFrame(models_comparison)
print(df_comp.to_string(index=False))

# Visualisation - Taille des modèles
model_names  = ['DistilBERT', 'BERT', 'RoBERTa', 'XLM-R', 'GPT-2', 'T5-Base']
model_params = [66, 110, 125, 125, 117, 220]  # en millions

plt.figure(figsize=(10, 5))
bars = plt.bar(model_names, model_params, color=['steelblue', 'orange', 'green', 'red', 'purple', 'brown'])
plt.title('Nombre de paramètres des principaux modèles Transformers (en millions)', fontsize=12)
plt.ylabel('Paramètres (millions)')
plt.xlabel('Modèle')
for bar, val in zip(bars, model_params):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val}M', ha='center', fontsize=10)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Guide de sélection des modèles
guide = {
    'Contrainte': ['Vitesse maximale', 'Meilleure précision (classif.)',
                   'Multilingue', 'Génération de texte',
                   'Résumé / Traduction', 'Peu de données'],
    'Modèle recommandé': ['DistilBERT / TinyBERT', 'RoBERTa-large',
                          'XLM-RoBERTa', 'GPT-2 / GPT-3.5',
                          'T5 / BART', 'GPT-3 (few-shot)'],
    'Raison': ['40-60% plus rapide que BERT', 'Fine-tuning optimisé',
               '100 langues, très robuste', 'Architecture decoder optimisée',
               'Encoder-Decoder, seq2seq', 'Zero/few-shot sans fine-tuning']
}
print('Guide de sélection des modèles :')
print(pd.DataFrame(guide).to_string(index=False))

## 11. Conclusion

In [ ]:
recap = {
    'Pipeline testé': ['text-classification', 'ner', 'question-answering',
                       'text-generation', 'fill-mask', 'summarization', 'zero-shot'],
    'Modèle': ['DistilBERT-SST2 (268MB)', 'BERT-large-CoNLL03 (1.33GB)',
               'DistilBERT-SQuAD (261MB)', 'GPT-2',
               'BERT-base', 'BART-large-CNN', 'BART-large-MNLI'],
    'Résultat': [
        'NEGATIVE — score: 0.9015',
        '10+ entités : Amazon(ORG), Germany(LOC), Optimus Prime(MISC)...',
        "'an exchange of Megatron' — score: 0.631",
        'Complétion cohérente de phrases',
        'Prédiction des mots masqués',
        'Résumé pertinent en 2-3 phrases',
        'Classification sans fine-tuning'
    ]
}
print('=== Récapitulatif des pipelines testés ===\n')
print(pd.DataFrame(recap).to_string(index=False))

print('\n=== Enseignements principaux ===')
print('• pipeline() permet d\'utiliser BERT/GPT state-of-the-art en 2-3 lignes de code')
print('• DistilBERT : 40% plus léger, 60% plus rapide, 97% des performances de BERT')
print('• Tokenisation sous-mot (WordPiece, BPE, SentencePiece) gère le vocabulaire ouvert')
print('• Mécanisme d\'attention : calcul Q, K, V → pondération de l\'importance de chaque token')
print('• 3 familles : Encoder-only (classif.), Decoder-only (génération), Enc-Dec (résumé/trad.)')

print('\n=== Perspectives ===')
print('• Fine-tuning de BERT sur dataset personnalisé (sentiment en arabe ou français)')
print('• Utilisation du pipeline summarization sur des articles scientifiques')
print('• Exploration de GPT-2 pour la génération de texte créatif')
print('• Visualisation des poids d\'attention avec BertViz')

print('\nBARHOINE AYOUB — Filière FIGI 2ème Année — Module : Deep Learning — TP Transformers NLP — 2026')